In [1]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
drive_root = "/content/drive/MyDrive"

pos_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [5]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [6]:
train_pos_df = pd.read_csv(pos_train_csv)
val_pos_df   = pd.read_csv(pos_val_csv)
test_pos_df  = pd.read_csv(pos_test_csv)

print("Train:", train_pos_df.shape)
print("Val  :", val_pos_df.shape)
print("Test :", test_pos_df.shape)

print(train_pos_df.head())
print(train_pos_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                  unit2_pos  unit1_beam
0   3532    [0.8092883966431671, 0.521083920903955]          17
1   2224  [0.4816276084988933, 0.29434536152734486]          14
2   9416    [0.220278556834608, 0.4136596156292844]          17
3   8510  [0.21412273613497904, 0.4547214157104936]          20
4   6877  [0.14500641727379412, 0.4097884695072434]          17
['index', 'unit2_pos', 'unit1_beam']


In [7]:
label_col = train_pos_df.columns[-1]
feature_cols = [c for c in train_pos_df.columns if c != label_col]

print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

Feature columns: ['index', 'unit2_pos']
Label column: unit1_beam
Label min/max: 2 30


In [8]:
train_mean = train_pos_df[feature_cols].astype(float).mean()
train_std  = train_pos_df[feature_cols].astype(float).std().replace(0, 1)

print("Mean:")
print(train_mean)

print("Std:")
print(train_std)

ValueError: could not convert string to float: '[0.8092883966431671, 0.521083920903955]'